# Sentiment Analysis on Tweets

This lab follows the authors one ("Text classification: finding a sentence's author"). We classify tweets into three categories, **negative**, **neutral** or **positive**, on the [TweetEval](https://huggingface.co/datasets/cardiffnlp/tweet_eval) dataset (*sentiment* task): about 60,000 English tweets, annotated by hand.

The text preparation, the recurrent classifier and its training with Lightning are those of the authors lab, under the same names (`Tokenizer`, `vectorize`, `TextGRU`, `TextClassifier`): they are given. This lab focuses on what changes:

1. Imbalanced classes, and the macro recall to measure them.
2. Batches without any *padding*: `pack_sequence`.
3. A bidirectional GRU that reads packed sequences.
4. Training and evaluating the GRU.
5. A transformer already fine-tuned on the task.
6. Fine-tuning a pretrained encoder yourself, or training its head only.

Parts 5 and 6 use transformers, the subject of the next chapter.

Remember to enable the GPU: Runtime > Change runtime type > GPU. Without it, the training of part 4 is several times slower, and the fine-tuning of part 6 takes hours.

## Installing and importing the libraries

In [ ]:
!pip install -q datasets lightning torchmetrics

In [ ]:
import collections
from collections.abc import Iterable

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import spacy
import torch
from datasets import load_dataset
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, recall_score
from torch import nn
from torch.nn.utils.rnn import PackedSequence, pack_sequence
from torch.utils.data import DataLoader
from torchmetrics.classification import MulticlassRecall

lightning.seed_everything(0)

## The data

Hugging Face's `datasets` library downloads the dataset already split into training, validation and test sets. The cell after it shows ten training tweets with their label, the class distribution and the length of the tweets (in words).

In [ ]:
tweets = load_dataset("cardiffnlp/tweet_eval", "sentiment")
label_names = ["negative", "neutral", "positive"]
print(tweets)

In [ ]:
train_df = tweets["train"].to_pandas()
train_df["sentiment"] = train_df.label.map(dict(enumerate(label_names)))
print(train_df.sample(10, random_state=0)[["sentiment", "text"]].to_string())

train_df.sentiment.value_counts().plot.bar(title="Class distribution")
plt.show()

train_df.text.str.split().str.len().plot.hist(bins=30, title="Tweet length (words)")
plt.show()

## 1. Imbalanced classes

How were user names handled? Are the classes balanced, and what does it imply for the choice of metric? What is the typical length of a tweet?

Your answer here

### Solution

- User names are replaced with `@user`: the tweets are anonymized.
- The *neutral* class is the most frequent, the *negative* class the rarest: accuracy would favor a model that mostly predicts *neutral*. TweetEval's official metric is the **per-class average recall** (*macro recall*), which gives the same weight to each class.
- A tweet is about twenty words long, rarely more than 40.

### A baseline: TF-IDF and logistic regression

Before training a neural network, let's measure what a simple model achieves: a logistic regression on a TF-IDF representation of the tweets. If the network doesn't do better, it's useless!

In [ ]:
vectorizer = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(tweets["train"]["text"])
X_val_tfidf = vectorizer.transform(tweets["validation"]["text"])

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_tfidf, tweets["train"]["label"])
baseline_recall = recall_score(
  tweets["validation"]["label"], baseline.predict(X_val_tfidf), average="macro"
)
print(f"TF-IDF validation macro recall: {baseline_recall:.3f}")

## The preparation of the authors lab

The `Tokenizer` class is the one of the authors lab, with one difference: it splits the text into words with spaCy (`spacy.blank("en")`, an English pipeline that only contains the tokenizer, very fast), which keeps punctuation and emoticons, which carry sentiment in a tweet, and splits contractions ("don't" → "do", "n't"). As in that lab, `tokenizer` learns the 20,000 words of its vocabulary from the training tweets only: `<pad>` (index 0, unused here), `<unk>` (index 1), then the words from the most to the least frequent.

`vectorize(texts)` gives the sequence of indices of each text, as in the authors lab, but with neither truncation nor *padding*: a list of tensors of different lengths, which part 2 packs. `train_data`, `val_data` and `test_data` are the lists of `(sequence of indices, label)` pairs.

In [ ]:
nlp = spacy.blank("en")


class Tokenizer:
  """Map the most frequent words of a corpus to indices, from 2 up."""

  def __init__(self, max_tokens: int) -> None:
    self.max_tokens = max_tokens
    self.vocabulary: list[str] = []
    self.word_index: dict[str, int] = {}

  def split(self, text: str) -> list[str]:
    # spaCy's tokenization: punctuation and emoticons kept
    return [token.lower_ for token in nlp.tokenizer(text) if not token.is_space]

  def fit_on_texts(self, texts: Iterable[str]) -> None:
    counts = collections.Counter(word for text in texts for word in self.split(text))
    # Indices 0 and 1 reserved for padding and unknown words
    most_common = [word for word, _ in counts.most_common(self.max_tokens - 2)]
    self.vocabulary = ["<pad>", "<unk>", *most_common]
    self.word_index = {word: index for index, word in enumerate(self.vocabulary)}

  def texts_to_sequences(self, texts: Iterable[str]) -> list[list[int]]:
    # A word outside the vocabulary becomes 1, <unk>
    return [
      [self.word_index.get(word, 1) for word in self.split(text)] for text in texts
    ]


tokenizer = Tokenizer(max_tokens=20_000)
tokenizer.fit_on_texts(tweets["train"]["text"])  # vocabulary of the training set only
vocabulary = tokenizer.vocabulary


def vectorize(texts: list[str]) -> list[torch.Tensor]:
  # Neither truncation nor padding. A tweet without any token becomes a
  # single <unk>: an empty sequence can't be packed
  return [
    torch.tensor(sequence or [1]) for sequence in tokenizer.texts_to_sequences(texts)
  ]


# (sequence of indices, label) pairs: a list is enough as a Dataset
train_data = list(zip(vectorize(tweets["train"]["text"]), tweets["train"]["label"]))
val_data = list(
  zip(vectorize(tweets["validation"]["text"]), tweets["validation"]["label"])
)
test_data = list(zip(vectorize(tweets["test"]["text"]), tweets["test"]["label"]))
print(" | ".join(vocabulary[:12]))
print(tweets["train"][0]["text"])
print(train_data[0])

## 2. Batches without *padding*: `pack_sequence`

Tweets don't all have the same length. Rather than padding the sequences of a batch, as `vectorize` does in the authors lab, PyTorch can **pack** them into a `PackedSequence`: all the tokens of the batch are laid end to end, time step by time step, without any *padding*.

- `data`: first the first token of each sequence, then the second token of each sequence that has at least two, etc.;
- `batch_sizes`: the number of sequences still running at each time step;
- `sorted_indices` / `unsorted_indices`: the order (by decreasing length) in which the sequences are laid out, and how to get back to the original order.

PyTorch's recurrent layers (`nn.RNN`, `nn.GRU`, `nn.LSTM`) accept a `PackedSequence` directly: at each time step, they only compute on the sequences still running, and their last hidden state `h_n` is the one of each sequence's last *real* token, put back in the original order. In the authors lab, `pack_padded_sequence` built this `PackedSequence` from a padded tensor; `pack_sequence` builds it directly from the sequences.

Run the next cell, then work out how `data`, `batch_sizes` and `sorted_indices` are built from the three sequences.

In [ ]:
example = pack_sequence(
  [torch.tensor([1, 2, 3]), torch.tensor([4, 5]), torch.tensor([6, 7, 8, 9])],
  enforce_sorted=False,
)
print(example)

Your answer here

### Solution

- `sorted_indices` is `[2, 0, 1]`: the sequences are laid out from the longest to the shortest, `[6, 7, 8, 9]` (no. 2), `[1, 2, 3]` (no. 0), then `[4, 5]` (no. 1).
- `data` is `[6, 1, 4, 7, 2, 5, 8, 3, 9]`: the first tokens of the three sequences, in that order, then their second tokens, then the third ones of the two sequences that have one, then the fourth one of the longest.
- `batch_sizes` is `[3, 3, 2, 1]`: the number of sequences still running at each of the four time steps. Its length is that of the longest sequence, and its sum the number of tokens.

### The `collate` function

Write the `collate(batch)` function the `DataLoader`s will use to form their batches: it receives a list of `(tensor of indices, label)` pairs, those of `train_data`, and returns the `(packed sequences, tensor of labels)` pair. The sequences of a batch aren't sorted by length: `pack_sequence` must take care of it.

**Useful documentation**:

- https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_sequence.html

In [ ]:
def collate(
  batch: list[tuple[torch.Tensor, int]],
) -> tuple[PackedSequence, torch.Tensor]:
  # Your code here
  ...

#### Additional hints

The calls and parameters to use:

- `sequences, labels = zip(*batch)`: a tuple of the sequences and a tuple of the labels
- `pack_sequence(sequences, enforce_sorted=False)`: `enforce_sorted=False` lets `pack_sequence` sort the sequences by length
- `torch.tensor(labels)`: the tensor of labels, which the cross-entropy expects
- `collate([(torch.tensor([1, 2, 3]), 0), (torch.tensor([4, 5]), 2)])` returns a `PackedSequence` with `data` `[1, 4, 2, 5, 3]` and `batch_sizes` `[2, 2, 1]`, and the labels `tensor([0, 2])`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def collate(
  batch: list[tuple[torch.Tensor, int]],
) -> tuple[PackedSequence, torch.Tensor]:
  sequences, labels = zip(*...)
  return pack_sequence(..., enforce_sorted=...), torch.tensor(...)


print(collate([(torch.tensor([1, 2, 3]), 0), (torch.tensor([4, 5]), 2)]))
```

#### Solution

In [ ]:
def collate(
  batch: list[tuple[torch.Tensor, int]],
) -> tuple[PackedSequence, torch.Tensor]:
  # A tuple of the sequences, a tuple of the labels
  sequences, labels = zip(*batch)
  # pack_sequence sorts the sequences by length itself
  return pack_sequence(sequences, enforce_sorted=False), torch.tensor(labels)


print(collate([(torch.tensor([1, 2, 3]), 0), (torch.tensor([4, 5]), 2)]))

### Packed batches

The `DataLoader`s form their batches with `collate`. On a training batch, let's compare the number of packed tokens with the size the same batch would have with *padding*:

In [ ]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_data, batch_size=64, collate_fn=collate)
test_loader = DataLoader(test_data, batch_size=64, collate_fn=collate)

packed, labels = next(iter(train_loader))
# Number of tweets × length of the longest
padded_size = len(labels) * len(packed.batch_sizes)
print(
  f"Tokens in the batch: {packed.data.numel()} packed against {padded_size} "
  f"with padding, so {1 - packed.data.numel() / padded_size:.0%} of padding avoided"
)

The lengths of the tweets in a batch vary a lot: with *padding*, a batch would hold about twice as many positions as real tokens, and the GRU would do as many useless computations. The `PackedSequence` avoids them.

## 3. A bidirectional GRU over packed sequences

`TextGRU` has the layers of the authors lab's one: `embedding`, a vector of 64 numbers per vocabulary word, a *dropout* of rate `dropout`, `gru`, a bidirectional GRU of `hidden_size` units per direction, and `head`, which gives a score (a logit) to each of the 3 classes from the last hidden states of both directions. But it receives a `PackedSequence` of indices, no longer a padded tensor: it no longer needs `pack_padded_sequence`.

Write its `forward(packed)` method, which returns the logits of the batch's tweets, of shape `(batch, 3)`, with the *dropout* on the embeddings and before `head`. `nn.Embedding` and `nn.Dropout` don't accept a `PackedSequence`: apply them to the tokens `packed.data`, then rebuild a `PackedSequence` with the same structure with `packed._replace(data=...)`, which keeps `batch_sizes` and the sorting indices.

Why `h_n[-2]` and `h_n[-1]`: what `h_n` holds at the end of a multi-layer, bidirectional GRU or LSTM, as a video: [`h_n` layer by layer](https://shuuchuu.github.io/videos/nn/en/gru-final-hidden-state.mp4).

**Useful documentation**:

- https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.PackedSequence.html
- https://docs.python.org/3/library/collections.html#collections.somenamedtuple._replace

In [ ]:
class TextGRU(nn.Module):
  def __init__(self, hidden_size: int = 64, dropout: float = 0.3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(vocabulary), 64, padding_idx=0)
    self.dropout = nn.Dropout(dropout)
    self.gru = nn.GRU(64, hidden_size, bidirectional=True)
    self.head = nn.Linear(2 * hidden_size, len(label_names))

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    # Your code here
    ...

### Additional hints

The calls and parameters to use:

- `packed.data`: the indices of all the batch's tokens, of shape `(tokens,)`; `self.embedding(packed.data)` has the shape `(tokens, 64)`
- `packed._replace(data=...)`: a new `PackedSequence`, with the same `batch_sizes` and sorting indices, and other `data`
- `_, h_n = self.gru(embedded)`: `h_n` has the shape `(2, batch, hidden_size)`, its rows in the original order of the tweets; `h_n[-2]` and `h_n[-1]` are the last states of the forward and backward directions
- `torch.cat([h_n[-2], h_n[-1]], dim=1)` puts them end to end, into a tensor of shape `(batch, 2 * hidden_size)`
- `TextGRU()(collate(train_data[:4])[0]).shape` is `torch.Size([4, 3])`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
class TextGRU(nn.Module):
  def __init__(self, hidden_size: int = 64, dropout: float = 0.3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(vocabulary), 64, padding_idx=0)
    self.dropout = nn.Dropout(dropout)
    self.gru = nn.GRU(64, hidden_size, bidirectional=True)
    self.head = nn.Linear(2 * hidden_size, len(label_names))

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    embedded = packed._replace(data=self.dropout(self.embedding(...)))
    _, h_n = self.gru(...)
    summary = torch.cat([..., ...], dim=1)
    return self.head(self.dropout(...))


print(TextGRU()(collate(train_data[:4])[0]).shape)
```

### Solution

In [ ]:
class TextGRU(nn.Module):
  """Embeddings, a bidirectional GRU over packed tweets, one score per class."""

  def __init__(self, hidden_size: int = 64, dropout: float = 0.3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(vocabulary), 64, padding_idx=0)
    self.dropout = nn.Dropout(dropout)
    self.gru = nn.GRU(64, hidden_size, bidirectional=True)
    self.head = nn.Linear(2 * hidden_size, len(label_names))

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    # Embeddings and dropout of all the batch's tokens at once, put back into
    # a PackedSequence with the same structure
    embedded = packed._replace(data=self.dropout(self.embedding(packed.data)))
    _, h_n = self.gru(embedded)
    # Last hidden states of the forward and backward directions, in the
    # original order of the tweets
    summary = torch.cat([h_n[-2], h_n[-1]], dim=1)
    return self.head(self.dropout(summary))


print(TextGRU()(collate(train_data[:4])[0]).shape)
print(TextGRU())
print(sum(p.numel() for p in TextGRU().parameters()), "parameters")

## 4. Training and evaluation

`TextClassifier` is the Lightning wrapper of the authors lab, which measures the per-class average recall (`MulticlassRecall(average="macro")`) in validation and test, instead of the accuracy. Its batches are `(PackedSequence, labels)` pairs: the `Trainer` moves them to the GPU itself, `PackedSequence` included.

The training is the one of the authors lab: at most 10 epochs, the gradient norm clipped to 1 (`gradient_clip_val`), against exploding gradients, `EarlyStopping` after 2 epochs without progress and `ModelCheckpoint` to keep the weights of the best epoch. One difference: the best epoch is the one with the best validation macro recall (`monitor="val_recall", mode="max"`), the task's metric, no longer the one with the lowest validation loss. `plot_metrics` then plots the learning curves.

In [ ]:
class TextClassifier(lightning.LightningModule):
  """Lightning wrapper: classify the sentiment of packed tweets."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-3) -> None:
    super().__init__()
    self.model = model
    self.learning_rate = learning_rate
    # torchmetrics needs one metric instance per stage
    self.recalls = nn.ModuleDict(
      {
        f"{stage}_recall": MulticlassRecall(
          num_classes=len(label_names), average="macro"
        )
        for stage in ("val", "test")
      }
    )

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    return self.model(packed)

  def step(
    self, batch: tuple[PackedSequence, torch.Tensor], stage: str
  ) -> torch.Tensor:
    packed, labels = batch
    logits = self(packed)
    loss = nn.functional.cross_entropy(logits, labels)
    # Without batch_size, Lightning would infer it from packed.data: the tokens
    self.log(
      f"{stage}_loss",
      loss,
      batch_size=len(labels),
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    if stage != "train":
      recall = self.recalls[f"{stage}_recall"]
      recall(logits, labels)
      self.log(
        f"{stage}_recall",
        recall,
        batch_size=len(labels),
        on_step=False,
        on_epoch=True,
        prog_bar=True,
      )
    return loss

  def training_step(
    self, batch: tuple[PackedSequence, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.step(batch, "train")

  def validation_step(
    self, batch: tuple[PackedSequence, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.step(batch, "val")

  def test_step(
    self, batch: tuple[PackedSequence, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.step(batch, "test")

  def predict_step(
    self, batch: tuple[PackedSequence, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    packed, _ = batch
    return self(packed).argmax(dim=1)

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


def plot_metrics(trainer: lightning.Trainer) -> None:
  """Plot the losses and the validation macro recall over the epochs."""
  # Training and validation metrics aren't written on the same lines of the
  # CSV file: we group them by epoch
  metrics = (
    pandas.read_csv(f"{trainer.logger.log_dir}/metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["train_loss", "val_loss"])
  )
  metrics[["train_loss", "val_loss"]].plot(title="Loss", marker="o")
  metrics[["val_recall"]].plot(
    title="Validation macro recall", marker="o", legend=False
  )
  plt.show()

In [ ]:
model = TextClassifier(TextGRU())
# Best epoch: the one with the best validation macro recall
checkpoint = ModelCheckpoint(monitor="val_recall", mode="max")
trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  gradient_clip_val=1.0,  # gradient norm clipping
  logger=CSVLogger("logs", name="sentiment_gru"),
  callbacks=[
    # tqdm's progress bar, rather than rich's, which Lightning picks when rich
    # is installed (Colab): it froze some trainings
    TQDMProgressBar(),
    EarlyStopping(monitor="val_recall", mode="max", patience=2),
    checkpoint,
  ],
)
trainer.fit(model, train_loader, val_loader)
print(f"Best validation macro recall: {checkpoint.best_model_score:.3f}")
plot_metrics(trainer)

### Evaluation on the test set

The next cell evaluates the best model on the test set, then shows, with `show_evaluation`, the classification report and the confusion matrix of its predictions and of the TF-IDF baseline's.

In [ ]:
def show_evaluation(
  truths: numpy.ndarray, predictions: numpy.ndarray, title: str
) -> None:
  """Show the classification report and the confusion matrix."""
  print(title)
  print(classification_report(truths, predictions, target_names=label_names, digits=3))
  ConfusionMatrixDisplay.from_predictions(
    truths, predictions, display_labels=label_names, normalize="true"
  )
  plt.title(title)
  plt.show()


trainer.test(model, test_loader, ckpt_path="best")

# trainer.predict returns the list of predict_step's outputs, one per batch
predictions = torch.cat(trainer.predict(model, test_loader, ckpt_path="best")).numpy()
truths = numpy.array(tweets["test"]["label"])
show_evaluation(truths, predictions, "Bidirectional GRU")

baseline_predictions = baseline.predict(vectorizer.transform(tweets["test"]["text"]))
show_evaluation(truths, baseline_predictions, "TF-IDF baseline")

### The confused classes

Which classes are most often confused? Does the GRU beat the TF-IDF baseline?

Your answer here

#### Solution

In our run:

- Confusions mostly involve the *neutral* class: the GRU takes 46% of the negative tweets and 32% of the positive ones for neutral; *negative* and *positive* are rarely confused with each other (less than 10%).
- The GRU trained from scratch is only slightly better than the TF-IDF baseline: a macro recall of 0.59 against 0.56 on the test set. It gains mostly on the *negative* class, the rarest in training, which the baseline classifies as neutral 60% of the time. With only a few tens of thousands of tweets, it lacks data to learn the meaning of words: this is the motivation for pretrained models.

## 5. A transformer already fine-tuned on the task

The [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) model is a RoBERTa pretrained on millions of tweets, then fine-tuned on this same task. The `transformers` `pipeline` function, which the next cell calls, makes it usable in one line.

Evaluate it on the first 1,000 test tweets, and compare its macro recall with the GRU's on the same tweets. What do you conclude?

**Useful documentation**:

- https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline

In [ ]:
from transformers import pipeline

classifier = pipeline(
  "sentiment-analysis",
  model="cardiffnlp/twitter-roberta-base-sentiment-latest",
  device=0 if torch.cuda.is_available() else -1,
)
print(classifier("I love this course on PyTorch!"))

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `tweets["test"].select(range(1000))`: the first 1,000 test tweets
- `classifier(list(subset["text"]), batch_size=64)`: a list of one dictionary per tweet, such as `{'label': 'positive', 'score': 0.98}`; the labels are `negative`, `neutral` and `positive`, in the order of `label_names`
- `recall_score(truths, predictions, average="macro")`: the macro recall; `predictions` holds the GRU's predictions on the whole test set, in order

A skeleton to copy into the exercise cell, replacing each `...`:

```python
subset = tweets["test"].select(range(...))
name_to_label = {"negative": 0, "neutral": 1, "positive": 2}
roberta_predictions = [
  name_to_label[output[...]]
  for output in classifier(list(subset["text"]), batch_size=64)
]
roberta_recall = recall_score(list(subset["label"]), ..., average="macro")
gru_recall = recall_score(truths[:1000], ...[:1000], average="macro")
print(f"RoBERTa: {roberta_recall:.3f}, GRU: {gru_recall:.3f}")
```

### Solution

In [ ]:
subset = tweets["test"].select(range(1000))
name_to_label = {"negative": 0, "neutral": 1, "positive": 2}
# The pipeline returns one dictionary per tweet: {'label': ..., 'score': ...}
roberta_predictions = [
  name_to_label[output["label"]]
  for output in classifier(list(subset["text"]), batch_size=64)
]
roberta_recall = recall_score(
  list(subset["label"]), roberta_predictions, average="macro"
)
gru_recall = recall_score(truths[:1000], predictions[:1000], average="macro")
print(f"RoBERTa: {roberta_recall:.3f}, GRU: {gru_recall:.3f}")

The pretrained transformer does much better (about 0.71 against 0.59 for the GRU): it learned the language of tweets from millions of examples before being fine-tuned. For a common text classification task, starting from a pretrained model is today's first reflex; training an RNN from scratch remains useful under tight compute constraints, or when no pretrained model exists for the language or domain.

## 6. Fine-tuning a pretrained encoder yourself

cardiffnlp's RoBERTa was fine-tuned by others on this task. Let's do it ourselves, from a general-purpose encoder that has never seen these labels: [`distilroberta-base`](https://huggingface.co/distilbert/distilroberta-base), a smaller version of RoBERTa (82M parameters). `AutoModelForSequenceClassification` adds a classification head with 3 outputs to it, with random weights (slide "Fine-Tuning a Pretrained Encoder", in the next chapter).

The next cell tokenizes the tweets with the encoder's tokenizer, and defines:

- `TransformerClassifier(name, learning_rate)`, the Lightning wrapper of the encoder and its head;
- `fine_tune(classifier, name)`, which trains a `TransformerClassifier` for one epoch, then returns its predictions on the first 1,000 test tweets, those of `truths[:1000]`.

In [ ]:
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  DataCollatorWithPadding,
)

ENCODER = "distilroberta-base"
encoder_tokenizer = AutoTokenizer.from_pretrained(ENCODER)
# Tokenization of each tweet: subword identifiers and attention mask
encoded = (
  tweets.map(
    lambda batch: encoder_tokenizer(batch["text"], truncation=True, max_length=128),
    batched=True,
  )
  .remove_columns(["text"])
  .rename_column("label", "labels")
)
# Padding is done batch by batch, to the length of the longest tweet
encoder_collate = DataCollatorWithPadding(encoder_tokenizer)
encoder_loaders = {
  split: DataLoader(
    encoded[split],
    batch_size=32,
    shuffle=split == "train",
    collate_fn=encoder_collate,
  )
  for split in ("train", "validation")
}
# The first 1,000 test tweets, those of the comparisons
encoder_loaders["test"] = DataLoader(
  encoded["test"].select(range(1000)), batch_size=32, collate_fn=encoder_collate
)


class TransformerClassifier(lightning.LightningModule):
  """Pretrained encoder topped with a new classification head."""

  def __init__(self, name: str, learning_rate: float = 2e-5) -> None:
    super().__init__()
    self.learning_rate = learning_rate
    # The classification head is created with random weights
    self.model = AutoModelForSequenceClassification.from_pretrained(
      name, num_labels=len(label_names)
    )
    # from_pretrained puts the model in evaluation mode, which Lightning 2.6 no
    # longer leaves at the start of fit: without train(), no dropout in training
    self.model.train()
    self.val_recall = MulticlassRecall(num_classes=len(label_names), average="macro")

  def step(self, batch: dict[str, torch.Tensor], stage: str) -> torch.Tensor:
    # With batch["labels"], the model computes the cross-entropy itself
    outputs = self.model(**batch)
    batch_size = len(batch["labels"])
    self.log(
      f"{stage}_loss",
      outputs.loss,
      batch_size=batch_size,
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    if stage == "val":
      self.val_recall(outputs.logits, batch["labels"])
      self.log(
        "val_recall",
        self.val_recall,
        batch_size=batch_size,
        on_step=False,
        on_epoch=True,
        prog_bar=True,
      )
    return outputs.loss

  def training_step(
    self, batch: dict[str, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.step(batch, "train")

  def validation_step(
    self, batch: dict[str, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.step(batch, "val")

  def predict_step(
    self, batch: dict[str, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.model(**batch).logits.argmax(dim=1)

  def configure_optimizers(self) -> torch.optim.Optimizer:
    # Frozen parameters (requires_grad_(False)) have no gradient: AdamW
    # leaves them as they are
    return torch.optim.AdamW(self.parameters(), lr=self.learning_rate)


def fine_tune(classifier: TransformerClassifier, name: str) -> numpy.ndarray:
  """Train classifier for one epoch, then predict the first 1,000 test tweets."""
  trainer = lightning.Trainer(
    max_epochs=1,
    accelerator="auto",
    devices=1,
    # Half-precision computations on a GPU: twice as fast
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    logger=CSVLogger("logs", name=name),
    # tqdm's progress bar, rather than rich's, which Lightning picks when rich
    # is installed (Colab): it froze some trainings
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,  # a single epoch, kept as is
  )
  trainer.fit(classifier, encoder_loaders["train"], encoder_loaders["validation"])
  return torch.cat(trainer.predict(classifier, encoder_loaders["test"])).cpu().numpy()

### Fine-tune the whole model, or train its head only?

A pretrained encoder is reused in two ways:

- **fine-tuning**: the whole model learns, with a small learning rate (2e-5) that preserves what the encoder learned during its pretraining;
- **feature extraction**: the encoder is frozen, and only the head learns, from its representations; it starts from scratch, with the learning rate of a new model (1e-3).

Compare them. Create two `TransformerClassifier`s, freeze the encoder of the second (`model.roberta`) with `requires_grad_`, and count the trainable parameters of each. Train them with `fine_tune`, then compare their macro recalls on the first 1,000 test tweets with those of the GRU and of cardiffnlp's RoBERTa. Which approach wins, and why? Which one has the shortest epoch?

**Useful documentation**:

- https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.requires_grad_

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `TransformerClassifier(ENCODER)`: full fine-tuning, with the default learning rate of 2e-5; `TransformerClassifier(ENCODER, learning_rate=1e-3)` for the head only
- `head_only.model.roberta.requires_grad_(False)` freezes all the encoder's parameters; the head is `head_only.model.classifier`
- `sum(p.numel() for p in transformer.parameters() if p.requires_grad)`: the number of trainable parameters, which the summary Lightning shows at the start of `fit` also gives
- the progress bar of `fit` shows the epoch's duration
- `fine_tune` returns the predictions of the first 1,000 test tweets; `predictions` and `roberta_predictions`: the predictions of the GRU (part 4) and of cardiffnlp's RoBERTa (part 5)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
full = TransformerClassifier(ENCODER)
head_only = TransformerClassifier(ENCODER, learning_rate=...)
head_only.model.roberta.requires_grad_(...)
for name, transformer in [("Full fine-tuning", full), ("Head only", head_only)]:
  trainable = sum(p.numel() for p in transformer.parameters() if ...)
  print(f"{name}: {trainable} trainable parameters")

full_predictions = fine_tune(full, "distilroberta_full")
head_predictions = fine_tune(..., "distilroberta_head")
for name, model_predictions in [
  ("GRU", predictions),
  ("DistilRoBERTa, head only", ...),
  ("Fine-tuned DistilRoBERTa", ...),
  ("cardiffnlp's RoBERTa", roberta_predictions),
]:
  score = recall_score(truths[:1000], model_predictions[:1000], average="macro")
  print(f"{name}: {score:.3f}")
```

#### Solution

In [ ]:
full = TransformerClassifier(ENCODER)
# A head starting from scratch: the learning rate of a new model
head_only = TransformerClassifier(ENCODER, learning_rate=1e-3)
# Frozen encoder: only the head, model.classifier, learns
head_only.model.roberta.requires_grad_(False)
for name, transformer in [("Full fine-tuning", full), ("Head only", head_only)]:
  trainable = sum(p.numel() for p in transformer.parameters() if p.requires_grad)
  print(f"{name}: {trainable} trainable parameters")

full_predictions = fine_tune(full, "distilroberta_full")
head_predictions = fine_tune(head_only, "distilroberta_head")
show_evaluation(truths[:1000], full_predictions, "Fine-tuned DistilRoBERTa")
print("Macro recall on the first 1,000 test tweets")
for name, model_predictions in [
  ("GRU", predictions),
  ("DistilRoBERTa, head only", head_predictions),
  ("Fine-tuned DistilRoBERTa", full_predictions),
  ("cardiffnlp's RoBERTa", roberta_predictions),
]:
  score = recall_score(truths[:1000], model_predictions[:1000], average="macro")
  print(f"{name}: {score:.3f}")

The head alone has only 592,899 trainable parameters, against 82,120,707 for full fine-tuning: backpropagation stops at the head, and its epoch is shorter.

But the fine-tuned encoder usually wins by far, and comes close to cardiffnlp's RoBERTa without matching it: in our run, on the first 1,000 test tweets, a macro recall of 0.593 for the GRU, 0.648 for the head alone, 0.710 for the fine-tuned encoder and 0.715 for cardiffnlp's RoBERTa. The representations of `distilroberta-base`, learned to guess masked words in general-purpose texts, aren't made for the sentiment of tweets: frozen, the head has to make do with them; fine-tuned, they adapt to the task. cardiffnlp's RoBERTa keeps the edge: larger, it was pretrained on millions of tweets, whose vocabulary and turns of phrase it knows.

Fine-tuning the whole encoder is the standard approach when no model exists for the task yet and a few thousand annotated examples and a GPU are available. Feature extraction remains useful with very little data, when compute is scarce, or to serve several tasks with the same frozen encoder.